In [1]:
import numpy as np
import pandas as pd
from dotenv import load_dotenv
import os

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.timeframe import TimeFrame
from alpaca.data.requests import StockBarsRequest
from alpaca.trading.client import TradingClient

In [2]:
#loading dotenv

load_dotenv()

KEY = os.getenv("key")
SECRET = os.getenv("secret")

#henting av data
client = StockHistoricalDataClient(KEY, SECRET)

df=client.get_stock_bars(StockBarsRequest(
    symbol_or_symbols="AAPL",
    timeframe=TimeFrame.Day,
    start="2026-01-01",
)).df

df.describe()


,open,high,low,close,volume,trade_count,vwap
count,168.000000,168.000000,168.000000,168.000000,1.680000e+02,1.680000e+02,168.000000
mean,284.528458,287.912090,281.595729,284.908214,5.013970e+07,7.899417e+05,284.840149
std,25.402083,25.967047,25.365206,25.879281,2.301418e+07,2.312505e+05,25.789290
min,247.320000,249.199900,243.420000,246.630000,2.452734e+07,4.825740e+05,246.869425
25%,259.670000,262.182500,257.670000,260.487500,3.896951e+07,6.478555e+05,260.420971
50%,278.492500,285.260000,275.680550,279.130000,4.517648e+07,7.542205e+05,280.311350
75%,307.895000,311.732500,305.687500,308.842500,5.333063e+07,8.539798e+05,309.228157
max,340.030000,344.569900,337.350100,340.080000,2.619466e+08,2.207523e+06,340.796598


In [42]:
from typing import Tuple
from alpaca.trading.enums import OrderSide

def sma_strategy(equity: int, df: pd.DataFrame, risk_pct: float = 0.01) -> Tuple[OrderSide, int, float]:
    sma50 = df['close'].rolling(window=50).mean().iloc[-1]
    sma200 = df['close'].rolling(window=200).mean().iloc[-1]

    high_low = df['high'] - df['low']
    high_close = abs(df['high'] - df['close'].shift())
    low_close = abs(df['low'] - df['close'].shift())

    tr = pd.concat([high_close, high_low, low_close], axis=1).max(axis=1)
    atr = tr.rolling(14).mean().iloc[-1]

    risk_per_share = 2*atr
    risk_amount = equity * risk_pct

    qty = int(risk_amount / risk_per_share)
    stop_loss = df['close'].iloc[-1] - risk_per_share

    if qty == 0:
        return None, 0, stop_loss

    if sma50 > sma200:
        max_qty = int(equity / df['close'].iloc[-1])
        qty = min(qty, max_qty)

        return OrderSide.BUY, qty, stop_loss

    elif sma50 < sma200:
        return OrderSide.SELL, qty, stop_loss

    else:
        return None, 0, stop_loss
    
    

In [43]:
#Back-testing the strategy
from datetime import date


start_cash = 10000
equity = start_cash #Eksempel på hvor mye egenkapital vi vil backteste på
positions = 0
entry_price = 0

trades = []
equity_curve = []

backtest_days = client.get_stock_bars(StockBarsRequest(
    symbol_or_symbols="AAPL",
    timeframe=TimeFrame.Day,
    start="2020-01-01",
    end="2025-01-01",
)).df

for day_i in range(200, len(backtest_days)):
    row = backtest_days.iloc[day_i]

    print("-"*30)
    # Bruk kun data frem til dagen før for å unngå look-ahead bias
    signal, qty, suggested_stop = sma_strategy(equity, backtest_days.iloc[:day_i])
    print(f"{day_i}: {signal}, {qty}")

    if positions == 0 and signal == OrderSide.BUY and qty > 0:
        print("Buying")
        positions = qty
        entry_price = row['close']
        stop_loss = suggested_stop
        equity -= qty * entry_price

        trades.append({
            "date": backtest_days.index[day_i],
            "day_i": day_i,
            "pnl": 0,
            "side": OrderSide.BUY,
            "qty": qty,
        })

    elif positions > 0:

        #Stop loss
        if row['low'] <= stop_loss:
            print("Stop loss CROSSED, selling")


            sold_qty = positions #for log-føring
            exit_price = min(row['open'], stop_loss)

            equity += sold_qty * exit_price
            pnl = (exit_price - entry_price) * positions
            positions = 0
            entry_price = None
            stop_loss = None

            trades.append({
                "date": backtest_days.index[day_i],
                "day_i": day_i,
                "pnl": pnl,
                "side": OrderSide.SELL,
                "qty": sold_qty,
            })

            #selLing
        elif signal == OrderSide.SELL:
            print("Selling")


            qty = min(positions, qty) 
            pnl = (row['open'] - entry_price) * qty
            equity += qty * row['open']
            positions -= qty
            
            trades.append({
                "date": backtest_days.index[day_i],
                "day_i": day_i,
                "pnl": pnl,
                "side": OrderSide.SELL,
                "qty": qty,
            })

            #Buying
        elif signal == OrderSide.BUY:
            print("Buying")

            entry_price = (entry_price * positions + row['open'] * qty) / (positions + qty) #gjennomsnittlig entry price
            positions += qty
            stop_loss = max(stop_loss, suggested_stop) #VElger strameste stop-loss
            equity -= qty * row['open']

            trades.append({
                "date": backtest_days.index[day_i],
                "day_i": day_i,
                "pnl": 0,
                "side": OrderSide.BUY,
                "qty": qty,
            })

        #holding
        else:
            print("Holding")

    portfolio_value = equity + positions * row['close']
    equity_curve.append(portfolio_value)

#Selge alle positions
if positions > 0:
    final_price = backtest_days.iloc[-1]['close']

    equity += positions * final_price

    trades.append({
        "date": backtest_days.index[-1],
        "day_i": len(backtest_days) - 1,
        "pnl": (final_price - entry_price) * positions,
        "side": OrderSide.SELL,
        "qty": positions,
    })

    postions = 0



            


            


        
        
        

------------------------------
200: OrderSide.SELL, 14
------------------------------
201: OrderSide.SELL, 14
------------------------------
202: OrderSide.SELL, 13
------------------------------
203: OrderSide.SELL, 13
------------------------------
204: OrderSide.SELL, 13
------------------------------
205: OrderSide.SELL, 13
------------------------------
206: OrderSide.SELL, 14
------------------------------
207: OrderSide.SELL, 14
------------------------------
208: OrderSide.SELL, 14
------------------------------
209: OrderSide.SELL, 13
------------------------------
210: OrderSide.SELL, 12
------------------------------
211: OrderSide.SELL, 12
------------------------------
212: OrderSide.SELL, 12
------------------------------
213: OrderSide.SELL, 13
------------------------------
214: OrderSide.SELL, 12
------------------------------
215: OrderSide.SELL, 12
------------------------------
216: OrderSide.SELL, 12
------------------------------
217: OrderSide.SELL, 12
----------

In [36]:
print(len(trades))
print(trades[:5])
print(set(t["side"] for t in trades))

210
[{'date': ('AAPL', Timestamp('2021-06-15 04:00:00+0000', tz='UTC')), 'day_i': 365, 'pnl': 0, 'side': <OrderSide.BUY: 'buy'>, 'qty': 26}, {'date': ('AAPL', Timestamp('2021-06-16 04:00:00+0000', tz='UTC')), 'day_i': 366, 'pnl': 0, 'side': <OrderSide.BUY: 'buy'>, 'qty': 18}, {'date': ('AAPL', Timestamp('2021-06-17 04:00:00+0000', tz='UTC')), 'day_i': 367, 'pnl': 0, 'side': <OrderSide.BUY: 'buy'>, 'qty': 11}, {'date': ('AAPL', Timestamp('2021-06-18 04:00:00+0000', tz='UTC')), 'day_i': 368, 'pnl': 0, 'side': <OrderSide.BUY: 'buy'>, 'qty': 7}, {'date': ('AAPL', Timestamp('2021-06-21 04:00:00+0000', tz='UTC')), 'day_i': 369, 'pnl': 0, 'side': <OrderSide.BUY: 'buy'>, 'qty': 4}]
{<OrderSide.SELL: 'sell'>, <OrderSide.BUY: 'buy'>}


In [44]:
# Resultat
trades_df = pd.DataFrame(trades)

print(f"PnL: {trades_df['pnl'].sum()}")
print(f"Final Value: {equity}")

total_return = (equity - start_cash) / start_cash
print(f"Total Return: {total_return:.2%}")

closed_trades = [
    t for t in trades
    if t["side"] == OrderSide.SELL
]

wins = [
    t for t in closed_trades
    if t["pnl"] > 0
]

win_rate = len(wins) / len(closed_trades)
print(f"Win Rate: {win_rate:.2%}")

peak = equity_curve[0]
max_drawdown = 0

for value in equity_curve:
    peak = max(peak, value)

    dd = (value - peak) / peak

    max_drawdown = min(max_drawdown, dd)

print(f"Max DD: {max_drawdown:.2%}")

returns = []

for i in range(1, len(equity_curve)):
    r = (
        equity_curve[i]
        / equity_curve[i-1]
        - 1
    )

    returns.append(r)

rf = 0      # risikofri rente

sharpe = (
    np.mean(returns) - rf/252
) / np.std(returns)

sharpe *= np.sqrt(252)

print(f"Sharpe Ratio: {sharpe:.2f}")

benchmark_return = (
    backtest_days.iloc[-1]["close"]
    /
    backtest_days.iloc[200]["close"]
    - 1
)

alpha = (
    total_return
    - benchmark_return
)

print(f"Alpha: {alpha:2%}")


PnL: 160.16055714283686
Final Value: 10160.160557142854
Total Return: 1.60%
Win Rate: 51.72%
Max DD: -34.60%
Sharpe Ratio: 0.11
Alpha: -108.800008%
